In [1]:
import random
import pandas as pd

# ==== ĐẶC TRƯNG NỀN ====
residence_area = {
    'high': ["TP Hồ Chí Minh", "Hà Nội", "Hải Phòng", "Quảng Ninh", "Đà Nẵng",
             "Lạng Sơn", "Lào Cai", "Tây Ninh", "Kiên Giang", "Bình Dương",
             "Đồng Nai", "Cần Thơ", "An Giang", "Bà Rịa - Vũng Tàu"],
    'medium': ["Bình Thuận", "Bình Phước", "Bình Định", "Khánh Hòa", "Nghệ An",
               "Thanh Hóa", "Thừa Thiên Huế", "Đắk Lắk", "Đắk Nông", "Gia Lai",
               "Kon Tum", "Quảng Nam", "Quảng Ngãi", "Phú Yên", "Hà Tĩnh",
               "Hải Dương", "Nam Định", "Ninh Bình", "Thái Nguyên", "Vĩnh Phúc",
               "Long An", "Hậu Giang", "Tiền Giang", "Trà Vinh", "Vĩnh Long",
               "Sóc Trăng", "Cà Mau", "Bạc Liêu", "Bến Tre", "Phú Thọ",
               "Hưng Yên", "Thái Bình", "Ninh Thuận", "Hòa Bình", "Yên Bái",
               "Tuyên Quang", "Hà Nam", "Lâm Đồng", "Quảng Bình"],
    'low': ["Bắc Giang", "Bắc Kạn", "Bắc Ninh", "Cao Bằng", "Điện Biên",
            "Hà Giang", "Sơn La", "Lai Châu", "Quảng Trị", "Đồng Tháp"]
}

occupation = {
    'high': ["Cầm đồ", "Kinh doanh nhà hàng karaoke", "Chủ quán bar", "Làm từ thiện",
             "Tiếp viên quán", "Vũ công tự do", "Streamer", "Youtuber", "Tiktoker",
             "Kinh doanh vàng bạc", "Chơi chứng khoán", "Doanh nhân", "Tự doanh",
             "Môi giới bất động sản", "Kinh doanh đa cấp", "Không rõ"],
    'medium': ["Luật sư", "Nhân viên ngân hàng", "Kế toán", "Kiểm toán viên",
               "Chuyên viên tài chính", "Tư vấn bảo hiểm", "Môi giới chứng khoán",
               "Tài xế giao dịch tiền", "Nhà báo", "Nghệ sĩ tự do", "Ca sĩ", "Diễn viên",
               "MC", "Freelancer", "Nhà văn", "Nhiếp ảnh gia", "Thiết kế đồ họa",
               "Chuyên gia SEO", "Quản trị fanpage", "Lái xe công nghệ", "Bán hàng online",
               "Chủ cửa hàng", "Lái taxi", "Chủ quán ăn", "Nhân viên marketing",
               "Cửa hàng trưởng", "Quản lý khách sạn", "Làm thuê thời vụ", "Trình dược viên",
               "Sinh viên", "Thất nghiệp", "Nội trợ"],
    'low': ["Công an", "Bộ đội", "Thẩm phán", "Kiểm sát viên", "Giảng viên đại học",
            "Giáo viên phổ thông", "Gia sư", "Nhà khoa học", "Kỹ sư phần mềm", "Lập trình viên",
            "Kỹ sư xây dựng", "Kỹ sư điện", "Kỹ thuật viên phòng lab", "Chuyên viên CNTT",
            "Phân tích dữ liệu", "Bác sĩ", "Y tá", "Dược sĩ", "Bác sĩ thú y", 
            "Nhân viên chăm sóc sắc đẹp", "Chuyên viên spa", "Chăm sóc người già", "Công nhân",
            "Thợ xây", "Thợ điện", "Thợ nước", "Thợ mộc", "Thợ hàn", "Lái xe tải", "Bốc vác",
            "Bảo vệ", "Nhân viên bảo trì", "Nhân viên bán hàng", "Nhân viên phục vụ", "Lễ tân",
            "Nhân viên thu ngân", "Tư vấn tuyển sinh", "Học sinh"]
}

def get_label(score):
    if score >= 17:
        return 4
    elif score >= 13:
        return 3
    elif score >= 9:
        return 2
    elif score >= 5:
        return 1
    else:
        return 0

def calculate_score(per_violation_score, per_role_score, per_legal, org_violation, org_legal, age, area, job):
    score = 0
    reasons = []

    # --- VI PHẠM CÁ NHÂN ---
    if per_legal == "Minh oan":
        reasons.append("Được minh oan (reset vi phạm và vai trò)")
    else:
        if per_violation_score > 0:
            score += per_violation_score
            reasons.append(f"Vi phạm: +{per_violation_score}")
        if per_role_score > 0:
            score += per_role_score
            reasons.append(f"Vai trò: +{per_role_score}")
        if per_legal == "Đã kết án":
            score += 3; reasons.append("Đã kết án (+3)")
        elif per_legal in ["Đang điều tra", "Truy tố"]:
            score += 2; reasons.append("Đang điều tra/truy tố (+2)")
        elif per_legal == "Chưa rõ":
            score += 1; reasons.append("Pháp lý chưa rõ (+1)")

    # --- VI PHẠM TỔ CHỨC ---
    if org_violation:
        if "tài chính" in org_violation or "cấm vận" in org_violation:
            score += 5; reasons.append("Tổ chức bị chế tài mạnh (+5)")
        elif "ngành" in org_violation:
            score += 4; reasons.append("Tổ chức bị hạn chế ngành (+4)")
        elif "thứ cấp" in org_violation:
            score += 3; reasons.append("Tổ chức bị trừng phạt thứ cấp (+3)")

    if org_legal == "Đã kết án":
        score += 3; reasons.append("Tổ chức đã kết án (+3)")
    elif org_legal in ["Đang điều tra", "Truy tố"]:
        score += 2; reasons.append("Tổ chức bị điều tra/truy tố (+2)")

    # --- ALIAS + TUỔI ---
    if age < 30 or age > 65:
        score += 1; reasons.append("Tuổi rủi ro (+1)")

    # --- ĐIỂM NỀN ---
    if area in residence_area['high']:
        score += 2; reasons.append("Khu vực rủi ro cao (+2)")
    elif area in residence_area['medium']:
        score += 1; reasons.append("Khu vực rủi ro vừa (+1)")

    if job in occupation['high']:
        score += 3; reasons.append("Nghề rủi ro cao (+3)")
    elif job in occupation['medium']:
        score += 1; reasons.append("Nghề rủi ro vừa (+1)")

    return score, reasons

def generate_aml_data_full_violation(n_samples=1000, seed=2025):
    random.seed(seed)
    samples = []
    unique_samples = set()
    area_list = [loc for sub in residence_area.values() for loc in sub]
    job_list = [job for sub in occupation.values() for job in sub]
    violation_type_pool = {
        6: ["Rửa tiền", "Tài trợ khủng bố"],
        4: ["Lừa đảo", "Chiếm đoạt tài sản", "Tham nhũng", "Hối lộ"],
        2: ["Vi phạm hành chính", "Tranh chấp dân sự", "Vi phạm dân sự", "Vi phạm nhỏ"],
        0: [None]
    }
    legal_pool = ["Đã kết án", "Đang điều tra", "Truy tố", "Chưa rõ", "Minh oan", None]
    role_pool = {
        6: ["Chủ mưu", "Cầm đầu", "Tổ chức thực hiện"],
        4: ["Tham gia", "Giúp sức", "Đồng phạm"],
        2: ["Bị nhắc tên", "Liên quan bị động"],
        0: [None]
    }
    org_violation_pool = [
        "Trừng phạt tài chính", "Cấm vận kinh tế",
        "Trừng phạt ngành", "Trừng phạt thứ cấp"
    ]
    org_legal_pool = ["Đã kết án", "Đang điều tra", "Truy tố", None]
    max_violation = 5
    target_distribution = {
        4: int(n_samples * random.uniform(0.04, 0.06)),
        3: int(n_samples * random.uniform(0.08, 0.12)),
        2: int(n_samples * random.uniform(0.15, 0.25)),
        1: int(n_samples * random.uniform(0.25, 0.35)),
    }
    target_distribution[0] = n_samples - sum(target_distribution.values())
    non_null_sample_count = 0

    for _ in range(n_samples * 15):  # tăng số vòng lặp để đảm bảo số lượng khi bị skip vì trùng
        area = random.choice(area_list)
        job = random.choice(job_list)
        age = int(random.randint(18, 75))

        # Đảm bảo 3 trường nền luôn có giá trị (tránh trường hợp có None)
        if area is None or job is None or age is None:
            continue

        per_violation_count = random.choices([1, 2, 3, 4, 5], weights=[0.7, 0.18, 0.07, 0.03, 0.02])[0]
        per_violation_scores = random.choices([6, 4, 2, 0], weights=[0.1, 0.15, 0.3, 0.45], k=per_violation_count)
        per_violation_types = []
        used_types = set()
        for score in per_violation_scores:
            candidates = [x for x in violation_type_pool[score] if x not in used_types]
            vtype = random.choice(candidates) if candidates else None
            per_violation_types.append(vtype)
            if vtype: used_types.add(vtype)
        while len(per_violation_types) < max_violation:
            per_violation_types.append(None)
        per_legal_statuses = [
            random.choices(
                legal_pool,
                weights=[0.03, 0.05, 0.04, 0.1, 0.01, 0.77]
            )[0] if vtype is not None else None
            for vtype in per_violation_types
        ]
        per_violation_score = per_violation_scores[0]
        per_violation = per_violation_types[0]
        per_legal = per_legal_statuses[0]
        if per_violation_score > 0:
            per_role_score = random.choices([6, 4, 2], weights=[0.3, 0.4, 0.3])[0]
            per_role = random.choice(role_pool[per_role_score])
        else:
            per_role_score = 0
            per_role = None
        org_violation_count = random.choices([0, 1, 2, 3, 4, 5], weights=[0.55, 0.25, 0.10, 0.06, 0.03, 0.01])[0]
        org_violation_types = []
        used_org_types = set()
        for _ in range(org_violation_count):
            candidates = [x for x in org_violation_pool if x not in used_org_types]
            vtype = random.choice(candidates) if candidates else None
            org_violation_types.append(vtype)
            if vtype: used_org_types.add(vtype)
        while len(org_violation_types) < max_violation:
            org_violation_types.append(None)
        org_legal_statuses = [
            random.choice(org_legal_pool) if vtype is not None else None
            for vtype in org_violation_types
        ]
        org_violation = org_violation_types[0]
        org_legal = org_legal_statuses[0]
        score, reasons = calculate_score(
            per_violation_score, per_role_score, per_legal,
            org_violation, org_legal, age, area, job
        )
        label = get_label(score)
        if sum(1 for s in samples if s["label"] == label) >= target_distribution[label]:
            continue
        is_non_null = any([
            per_violation_score > 0,
            per_role_score > 0,
            per_legal not in ("", None),
            org_violation not in ("", None),
            org_legal not in ("", None)
        ])
        if not is_non_null and non_null_sample_count < int(n_samples * 0.7):
            continue
        elif is_non_null:
            non_null_sample_count += 1

        row = {
            "residence_area": area,
            "occupation": job,
            "age": age,
            "per_role": per_role,
            "total_score": score,
            "label": label,
            "risk_reason": "; ".join(reasons)
        }
        # Add per_violation_type_x, per_legal_status_x
        for i in range(max_violation):
            row[f"per_violation_type_{i+1}"] = per_violation_types[i]
            row[f"per_legal_status_{i+1}"] = per_legal_statuses[i]
        # Add org_violation_type_x, org_legal_status_x
        for i in range(max_violation):
            row[f"org_violation_type_{i+1}"] = org_violation_types[i]
            row[f"org_legal_status_{i+1}"] = org_legal_statuses[i]

        sample_tuple = tuple(row.values())
        if sample_tuple in unique_samples:
            continue  # Bỏ qua nếu trùng
        unique_samples.add(sample_tuple)
        samples.append(row)
        if len(samples) >= n_samples:
            break

    df = pd.DataFrame(samples)
    # Kiểm tra cuối cùng để đảm bảo 3 trường luôn có giá trị
    assert df['residence_area'].isnull().sum() == 0
    assert df['occupation'].isnull().sum() == 0
    assert df['age'].isnull().sum() == 0
    return df


# Label
- 0: AML rất thấp
- 1: AML thấp
- 2: AML trung bình
- 3: AML cao
- 4: AML rất cao

In [2]:
import random
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

In [3]:
df = generate_aml_data_full_violation(1000, seed=42)

In [4]:
import pandas as pd

pd.set_option('display.max_columns', None)

# pd.set_option('display.max_rows', None)

pd.set_option('display.expand_frame_repr', False)

pd.set_option('display.width', 200)

df.head(50)

,residence_area,occupation,age,per_role,total_score,label,risk_reason,per_violation_type_1,per_legal_status_1,per_violation_type_2,per_legal_status_2,per_violation_type_3,per_legal_status_3,per_violation_type_4,per_legal_status_4,per_violation_type_5,per_legal_status_5,org_violation_type_1,org_legal_status_1,org_violation_type_2,org_legal_status_2,org_violation_type_3,org_legal_status_3,org_violation_type_4,org_legal_status_4,org_violation_type_5,org_legal_status_5
0,Hòa Bình,Môi giới bất động sản,61,Tổ chức thực hiện,14,3,Vi phạm: +2; Vai trò: +6; Đang điều tra/truy t...,Vi phạm hành chính,Truy tố,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,An Giang,Nhân viên thu ngân,62,Chủ mưu,18,4,Vi phạm: +4; Vai trò: +6; Tổ chức bị hạn chế n...,Tham nhũng,None,None,None,None,None,None,None,None,None,Trừng phạt ngành,Truy tố,None,None,None,None,None,None,None,None
2,Bình Dương,Diễn viên,66,Giúp sức,20,4,Vi phạm: +6; Vai trò: +4; Tổ chức bị trừng phạ...,Rửa tiền,None,None,None,None,None,None,None,None,None,Trừng phạt thứ cấp,Đã kết án,Trừng phạt ngành,None,None,None,None,None,None,None
3,Bến Tre,Quản trị fanpage,62,None,4,0,Tổ chức bị điều tra/truy tố (+2); Khu vực rủi ...,None,None,None,None,Hối lộ,None,None,None,None,None,Cấm vận kinh tế,Đang điều tra,Trừng phạt thứ cấp,Đã kết án,Trừng phạt ngành,Truy tố,Trừng phạt tài chính,None,None,None
4,Kiên Giang,Dược sĩ,49,Tổ chức thực hiện,15,3,Vi phạm: +6; Vai trò: +6; Pháp lý chưa rõ (+1)...,Rửa tiền,Chưa rõ,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
5,Đà Nẵng,Bộ đội,42,Bị nhắc tên,13,3,Vi phạm: +2; Vai trò: +2; Tổ chức bị chế tài m...,Vi phạm dân sự,None,None,None,None,None,None,None,None,None,Trừng phạt tài chính,Truy tố,None,None,None,None,None,None,None,None
6,Tuyên Quang,Lễ tân,39,Liên quan bị động,10,2,Vi phạm: +2; Vai trò: +2; Đã kết án (+3); Tổ c...,Vi phạm nhỏ,Đã kết án,None,None,None,None,None,None,None,None,Cấm vận kinh tế,Đang điều tra,Trừng phạt thứ cấp,Đang điều tra,Trừng phạt tài chính,Truy tố,Trừng phạt ngành,Đang điều tra,None,None
7,Long An,Nhân viên chăm sóc sắc đẹp,18,Liên quan bị động,6,1,Vi phạm: +2; Vai trò: +2; Tuổi rủi ro (+1); Kh...,Vi phạm hành chính,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
8,Bình Phước,Thợ điện,23,Tham gia,8,1,Vi phạm: +2; Vai trò: +4; Tuổi rủi ro (+1); Kh...,Vi phạm hành chính,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
9,Nam Định,Công nhân,28,None,4,0,Tổ chức bị điều tra/truy tố (+2); Tuổi rủi ro ...,None,None,None,None,None,None,None,None,None,None,Cấm vận kinh tế,Truy tố,Trừng phạt thứ cấp,None,Trừng phạt ngành,None,Trừng phạt tài chính,Đã kết án,None,None


In [5]:
categorical_features = [
    'residence_area', 'occupation', 'per_role',  # thêm các cột categorical khác nếu có
    'per_violation_type_1', 'per_legal_status_1', 'per_violation_type_2', 'per_legal_status_2',
    'per_violation_type_3', 'per_legal_status_3', 'per_violation_type_4', 'per_legal_status_4',
    'per_violation_type_5', 'per_legal_status_5',
    'org_violation_type_1', 'org_legal_status_1', 'org_violation_type_2', 'org_legal_status_2',
    'org_violation_type_3', 'org_legal_status_3', 'org_violation_type_4', 'org_legal_status_4',
    'org_violation_type_5', 'org_legal_status_5'
]

In [6]:
X = df.drop(['label', 'risk_reason', 'total_score'], axis=1)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [7]:
for col in categorical_features:
    X_train[col] = X_train[col].fillna("Unknown")
    X_test[col] = X_test[col].fillna("Unknown")

In [8]:
import pandas as pd

# Kiểm tra các cột có None/NaN
cols_with_nan = X_train.columns[X_train.isnull().any()]
print("Các cột có None/NaN trong X_train:", list(cols_with_nan))

for col in cols_with_nan:
    nan_count = X_train[col].isnull().sum()
    print(f"Cột '{col}' có {nan_count} giá trị NaN hoặc None")


Các cột có None/NaN trong X_train: []


In [9]:
cols_with_nan_test = X_test.columns[X_test.isnull().any()]
print("Các cột có None/NaN trong X_test:", list(cols_with_nan_test))

Các cột có None/NaN trong X_test: []


In [10]:
print(X_train.dtypes)

residence_area          object
occupation              object
age                      int64
per_role                object
per_violation_type_1    object
per_legal_status_1      object
per_violation_type_2    object
per_legal_status_2      object
per_violation_type_3    object
per_legal_status_3      object
per_violation_type_4    object
per_legal_status_4      object
per_violation_type_5    object
per_legal_status_5      object
org_violation_type_1    object
org_legal_status_1      object
org_violation_type_2    object
org_legal_status_2      object
org_violation_type_3    object
org_legal_status_3      object
org_violation_type_4    object
org_legal_status_4      object
org_violation_type_5    object
org_legal_status_5      object
dtype: object


In [11]:
num_cols = X_train.select_dtypes(include=['number']).columns
for col in num_cols:
    if X_train[col].isnull().any():
        print(f"Cột số '{col}' có NaN!")

In [12]:
for col in cols_with_nan:
    print(X_train[[col]].dropna().head())

In [13]:
y_train.head()

393    1
704    0
174    1
128    1
952    0
Name: label, dtype: int64

In [14]:
from catboost import CatBoostClassifier, Pool

train_pool = Pool(X_train, y_train, cat_features=categorical_features)
test_pool = Pool(X_test, cat_features=categorical_features)

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=42,
    verbose=50
)
model.fit(train_pool)

y_pred = model.predict(test_pool)
print(classification_report(y_test, y_pred))

0:	learn: 0.6912500	total: 85.9ms	remaining: 25.7s
50:	learn: 0.8250000	total: 2.6s	remaining: 12.7s
100:	learn: 0.8762500	total: 5.35s	remaining: 10.5s
150:	learn: 0.9150000	total: 8.12s	remaining: 8.01s
200:	learn: 0.9312500	total: 10.9s	remaining: 5.35s
250:	learn: 0.9575000	total: 13.7s	remaining: 2.67s
299:	learn: 0.9650000	total: 16.4s	remaining: 0us
              precision    recall  f1-score   support

           0       0.95      0.94      0.95        84
           1       0.80      0.83      0.82        54
           2       0.64      0.75      0.69        36
           3       0.55      0.38      0.44        16
           4       0.88      0.70      0.78        10

    accuracy                           0.82       200
   macro avg       0.76      0.72      0.74       200
weighted avg       0.82      0.82      0.82       200



In [15]:
import joblib
joblib.dump(model, 'catboost_aml_model.pkl')

['catboost_aml_model.pkl']

In [16]:
import joblib
model = joblib.load('catboost_aml_model.pkl')

# Inference

In [ ]:
sample = {
    "residence_area": "Hà Nội",
    "occupation": "Doanh nhân",
    "age": 38,
    
    "per_violation_type_1": "Rửa tiền",
    "per_legal_status_1": "Đã kết án",
    "per_role_1": "Chủ mưu",

    "per_violation_type_2": "Vi phạm hành chính",
    "per_legal_status_2": "Chưa rõ",
    "per_role_2": "Bị nhắc tên",

    "per_violation_type_3": None,
    "per_legal_status_3": None,
    "per_role_3": None,

    "per_violation_type_4": None,
    "per_legal_status_4": None,
    "per_role_4": None,

    "per_violation_type_5": None,
    "per_legal_status_5": None,
    "per_role_5": None,

    "org_violation_type_1": "Trừng phạt tài chính",
    "org_legal_status_1": "Đang điều tra",
    "org_role_1": "Chủ quản",

    "org_violation_type_2": None,
    "org_legal_status_2": None,
    "org_role_2": None,

    "org_violation_type_3": None,
    "org_legal_status_3": None,
    "org_role_3": None,

    "org_violation_type_4": None,
    "org_legal_status_4": None,
    "org_role_4": None,

    "org_violation_type_5": None,
    "org_legal_status_5": None,
    "org_role_5": None
}

import pandas as pd
for k, v in sample.items():
    if v is None:
        sample[k] = "None"

sample_df = pd.DataFrame([sample])

In [18]:
sample_pred = model.predict(sample_df)
print("Label dự đoán cho sample:", int(sample_pred[0]))

Label dự đoán cho sample: 4


/tmp/ipykernel_19/108674422.py:2: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print("Label dự đoán cho sample:", int(sample_pred[0]))
